# 02. AI 하중 예측 모델

01에서 만든 물리식 기준선(논문 M3, R² 0.944, MAE 1.27 MN)보다 더 잘 맞히는 AI 모델을 만들어요.

### 만드는 모델 3개

| 모델 | 어떻게 배우나 | 도구 |
|---|---|---|
| **GBDT** (경사 부스팅) | 작은 결정 나무를 수백 개 차례로 만들어요. 새 나무는 앞 나무들이 틀린 만큼을 고치도록 배워요. 표 데이터에 강해요 | scikit-learn `HistGradientBoostingRegressor` |
| **MLP** (신경망) | 숫자를 받아 여러 층을 거쳐 하중을 내는 신경망이에요. 경사하강법으로 가중치를 조금씩 고쳐 가며 배워요 | PyTorch |
| **하이브리드** | 논문 M3 식으로 먼저 하중을 계산하고, GBDT가 "M3가 몇 % 틀리는지"를 배워서 곱해 줘요. 물리식의 뼈대를 지키면서 AI가 빈틈을 메워요 | M3 × GBDT 보정 |

### 입력과 평가
- **입력:** 강종, 스탠드 번호, 입측·출측 두께, 폭, 롤 반경, 온도 + 이 값들로 계산한 물리 변수(압하율, 압하량, 접촉 길이, 변형률, 변형 속도)
- **넣지 않는 것:** `k_mean`(하중으로 계산한 값이라 정답이 섞여 있음), 코일 번호, 연주기
- **평가:** 01과 **똑같은 코일 단위 5-fold**예요. 모든 행이 한 번씩 "처음 보는 코일"로 예측돼서, 01의 숫자와 바로 비교할 수 있어요.

## 0. 준비

01과 같은 방식으로 데이터를 불러오고(pjh 정리 기준), 물리 변수와 5-fold를 만들어요.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

installed = {f.name for f in font_manager.fontManager.ttflist}
for font in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if font in installed:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False

BLUE, BLUE_DARK, GRAY = "#2a78d6", "#184f95", "#b5b3ab"
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlesize": 12,
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})

STANDS = ["F1", "F2", "F3", "F4", "F5"]

# 데이터: pjh 정리 기준 (01과 같음)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pjh/data_utils.py").exists())
sys.dont_write_bytecode = True   # 다른 팀원 폴더에 캐시(__pycache__)를 만들지 않음
sys.path.insert(0, str(REPO / "pjh"))
sys.path.insert(0, str(REPO / "kdh/src"))
from data_utils import load_clean                     # pjh 데이터 정리 기준 (팀 공통)
from prepare_data import COLS, add_features           # kdh 열 이름·파생 변수

clean, clean_info = load_clean()                      # 스탠드 누락·두께 불연속·폭 변동 코일 제외
df = add_features(clean.rename(columns=COLS))
df["chain_ok"] = True                                 # 정리 후 남은 코일은 모두 연속

# 물리 변수 (01과 같은 식). 나중에 두께를 바꿔 다시 계산할 수 있게 함수로 만듦
V1, HO1 = 1096.0, 27.7

def add_physics(d):
    d = d.copy()
    d["dh"] = d["h_in"] - d["h_out"]
    d["reduction"] = d["dh"] / d["h_in"]
    d["L"] = np.sqrt(d["R"] * d["dh"])
    d["strain"] = np.log(d["h_in"] / d["h_out"])
    d["v"] = V1 * HO1 / d["h_out"]
    d["strain_rate"] = d["strain"] * d["v"] / d["L"]
    d["area"] = d["W"] * d["L"]
    return d

df = add_physics(df)

# 코일 단위 5-fold (01과 같은 코드 → 같은 묶음)
df["fold"] = -1
for k, (_, test_idx) in enumerate(GroupKFold(n_splits=5).split(df, groups=df["coil"])):
    df.loc[df.index[test_idx], "fold"] = k

y = df["F_MN"]
print(f"{len(df):,}행, 코일 {df['coil'].nunique():,}개, fold별 행 수 {df['fold'].value_counts().sort_index().tolist()}")

### 입력 변수 정하기

In [ ]:
NUM = ["stand_no", "h_in", "h_out", "W", "R", "T_K", "reduction", "dh", "L", "strain", "strain_rate"]
FEATURES = ["grade"] + NUM
GRADES = sorted(df["grade"].unique())


def tree_input(d):
    """GBDT 입력: 강종은 범주형(category)으로 표시하면 GBDT가 알아서 나눠 씀"""
    X = d[FEATURES].copy()
    X["grade"] = pd.Categorical(X["grade"], categories=GRADES)
    return X

print("입력 변수", len(FEATURES), "개:", FEATURES)

### 두께를 조금 더 누른 가상 데이터 (최적화 대비 점검용)

최적화는 "이 스탠드에서 조금 더 누르면 하중이 어떻게 되나"를 수없이 물어봐요. 그래서 모델이 **더 누르면 하중이 커진다**는 당연한 성질을 지키는지 미리 봐요.
각 행의 출측 두께를 압하량의 5%만큼 더 얇게 만든 가상 데이터를 만들고, 원래 예측보다 하중이 **줄어드는** 행이 몇 %인지 셀 거예요.

In [ ]:
df_more = df.copy()
df_more["h_out"] = df["h_out"] - 0.05 * df["dh"]   # 압하량 5% 더 누름
df_more = add_physics(df_more)
print(f"압하율 평균: 원래 {df['reduction'].mean():.3f} → 더 누름 {df_more['reduction'].mean():.3f}")

## 1. 물리 기준선 다시 계산 (비교용)

01과 같은 M3 공통(기준선)과 M3 강종별(참고)을 같은 fold로 다시 계산해 둬요. 하이브리드 모델의 뼈대로도 써요.

In [ ]:
def fit_m3(d):
    y_log = np.log(d["F_N"] / d["area"])
    n, lnA = np.polyfit(np.log(d["strain_rate"]), y_log, 1)
    return np.exp(lnA), n

def predict_m3(d, A, n):
    return A * d["strain_rate"] ** n * d["area"] / 1e6   # MN

MODELS = ["M3 (기준선)", "M3 강종별", "GBDT", "MLP", "하이브리드"]
pred = pd.DataFrame(index=df.index, columns=MODELS, dtype=float)       # 원래 데이터 예측
pred_more = pd.DataFrame(index=df.index, columns=MODELS, dtype=float)  # 더 누른 데이터 예측

for k in range(5):
    train, test = df[df["fold"] != k], df[df["fold"] == k]
    A, n = fit_m3(train)
    for d, out in [(df, pred), (df_more, pred_more)]:
        out.loc[test.index, "M3 (기준선)"] = predict_m3(d.loc[test.index], A, n)
    for grade, train_g in train.groupby("grade"):
        A_g, n_g = fit_m3(train_g)
        idx = test.index[test["grade"] == grade]
        for d, out in [(df, pred), (df_more, pred_more)]:
            out.loc[idx, "M3 강종별"] = predict_m3(d.loc[idx], A_g, n_g)

print("M3 기준선 R²:", round(r2_score(y, pred["M3 (기준선)"]), 3))

## 2. GBDT

- `max_iter=600`: 나무를 최대 600개까지 쌓아요. `early_stopping`이 켜져 있어서, 학습 데이터 일부(10%)로 확인하다가 더 나아지지 않으면 일찍 멈춰요.
- `learning_rate=0.05`: 나무 하나가 고치는 정도. 작을수록 천천히, 꼼꼼하게 배워요.
- `max_leaf_nodes=31`: 나무 하나의 크기(잎 31개).

In [ ]:
def make_gbdt():
    return HistGradientBoostingRegressor(
        max_iter=600, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
        l2_regularization=1.0, categorical_features="from_dtype",
        early_stopping=True, n_iter_no_change=30, random_state=0,
    )

start = time.time()
for k in range(5):
    train, test = df[df["fold"] != k], df[df["fold"] == k]
    model = make_gbdt().fit(tree_input(train), train["F_MN"])
    pred.loc[test.index, "GBDT"] = model.predict(tree_input(test))
    pred_more.loc[test.index, "GBDT"] = model.predict(tree_input(df_more.loc[test.index]))
    print(f"fold {k}: 나무 {model.n_iter_}개, MAE {mean_absolute_error(test['F_MN'], pred.loc[test.index, 'GBDT']):.3f} MN")
print(f"GBDT R² {r2_score(y, pred['GBDT']):.3f}, 걸린 시간 {time.time() - start:.0f}초")

## 3. MLP (PyTorch 신경망)

- **입력 손질:** 신경망은 숫자 크기가 제각각이면(폭 1,300 vs 변형률 0.3) 잘 못 배워요. 그래서 학습 데이터의 평균·표준편차로 모든 숫자를 비슷한 크기로 맞추고(표준화), 강종은 0/1 열 10개로 바꿔요(원-핫).
- **구조:** 입력 21개 → 128 → 64 → 1(하중). 층 사이에 ReLU를 넣어 곡선 관계도 배울 수 있게 해요.
- **학습:** 256행씩 묶어서(배치) 오차를 계산하고, Adam(경사하강법의 한 종류)으로 가중치를 고쳐요. 데이터 전체를 60번 훑어요(에폭 60).
- fold 5개를 모두 학습하는 데 30초 안팎 걸려요.

In [ ]:
torch.set_num_threads(1)   # 작은 신경망은 CPU 스레드 1개가 가장 빠름 (여러 개면 오히려 느려짐)

def mlp_input(d, mean, std):
    num = (d[NUM] - mean) / std
    onehot = pd.get_dummies(pd.Categorical(d["grade"], categories=GRADES)).astype(float)
    return torch.tensor(np.hstack([num.to_numpy(), onehot.to_numpy()]), dtype=torch.float32)


def train_mlp(X, y_std, epochs=60, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(X.shape[1], 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # 갈수록 보폭을 줄임
    loader = DataLoader(TensorDataset(X, y_std), batch_size=256, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    losses = []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = nn.functional.mse_loss(model(xb).squeeze(1), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        scheduler.step()
        losses.append(total / len(X))
    model.eval()
    return model, losses


start = time.time()
for k in range(5):
    train, test = df[df["fold"] != k], df[df["fold"] == k]
    mean, std = train[NUM].mean(), train[NUM].std()
    y_mean, y_sd = train["F_MN"].mean(), train["F_MN"].std()
    y_std = torch.tensor(((train["F_MN"] - y_mean) / y_sd).to_numpy(), dtype=torch.float32)

    model, losses = train_mlp(mlp_input(train, mean, std), y_std)
    with torch.no_grad():
        for d, out in [(df, pred), (df_more, pred_more)]:
            out.loc[test.index, "MLP"] = model(mlp_input(d.loc[test.index], mean, std)).squeeze(1).numpy() * y_sd + y_mean
    if k == 0:
        first_losses = losses
    print(f"fold {k}: 마지막 학습 오차 {losses[-1]:.4f}, MAE {mean_absolute_error(test['F_MN'], pred.loc[test.index, 'MLP']):.3f} MN")
print(f"MLP R² {r2_score(y, pred['MLP']):.3f}, 걸린 시간 {time.time() - start:.0f}초")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(range(1, len(first_losses) + 1), first_losses, color=BLUE, linewidth=2)
ax.set_xlabel("에폭 (데이터를 훑은 횟수)")
ax.set_ylabel("학습 오차 (표준화 MSE)")
ax.set_title("MLP 학습 곡선 (fold 0)")
plt.tight_layout()
plt.show()

## 4. 하이브리드 (M3 × GBDT 보정)

1. fold마다 학습 데이터로 M3 계수를 맞춰 물리 예측값을 계산해요.
2. GBDT는 하중 자체가 아니라 **"실제 ÷ M3 예측"의 로그**(M3가 몇 % 틀리는지)를 배워요.
3. 최종 예측 = M3 예측 × GBDT가 알려 준 보정 배율

In [ ]:
start = time.time()
for k in range(5):
    train, test = df[df["fold"] != k], df[df["fold"] == k]
    A, n = fit_m3(train)
    target = np.log(train["F_MN"] / predict_m3(train, A, n))   # M3가 틀린 비율(로그)
    model = make_gbdt().fit(tree_input(train), target)
    for d, out in [(df, pred), (df_more, pred_more)]:
        t = d.loc[test.index]
        out.loc[test.index, "하이브리드"] = predict_m3(t, A, n) * np.exp(model.predict(tree_input(t)))
    print(f"fold {k}: MAE {mean_absolute_error(test['F_MN'], pred.loc[test.index, '하이브리드']):.3f} MN")
print(f"하이브리드 R² {r2_score(y, pred['하이브리드']):.3f}, 걸린 시간 {time.time() - start:.0f}초")

## 5. 성능 비교표

R²는 1에 가까울수록, MAE·RMSE·오차율은 작을수록 좋아요. **기준선 대비**는 MAE가 M3 기준선보다 몇 % 줄었는지예요.

In [ ]:
assert pred.notna().all().all()

def scores(p):
    return pd.Series({
        "R²": r2_score(y, p),
        "MAE (MN)": mean_absolute_error(y, p),
        "RMSE (MN)": root_mean_squared_error(y, p),
        "평균 오차율 (%)": ((p - y).abs() / y).mean() * 100,
    })

result = pd.DataFrame({m: scores(pred[m]) for m in MODELS}).T
base_mae = result.loc["M3 (기준선)", "MAE (MN)"]
result["기준선 대비 MAE"] = (result["MAE (MN)"] / base_mae - 1) * 100
display(result.style.format({"R²": "{:.3f}", "MAE (MN)": "{:.2f}", "RMSE (MN)": "{:.2f}",
                             "평균 오차율 (%)": "{:.1f}", "기준선 대비 MAE": "{:+.0f}%"}))

- **AI 모델 셋 다 기준선(M3, R² 0.944, MAE 1.27 MN)을 크게 넘었어요.**
  - 하이브리드: R² 0.992, MAE 0.45 MN, 평균 오차율 2.5% → **MAE 64% 감소, 가장 좋음**
  - GBDT: R² 0.991, MAE 0.46 MN → 63% 감소. 하이브리드와 0.01 MN 차이라 사실상 같아요.
  - MLP: R² 0.988, MAE 0.55 MN → 57% 감소. 셋 중에서는 조금 뒤져요.
- 물리식을 강종별로 맞춘 것(M3 강종별, MAE 1.05 MN)보다도 오차가 절반 이하예요. 물리식 두 변수(변형 속도, 면적)만으로는 못 잡는 패턴을 AI가 잡았다는 뜻이에요.
- 모든 숫자는 **처음 보는 코일**에 대한 성능이에요(코일 단위 5-fold).

## 6. 스탠드별·강종별 평균 오차율 (%)

In [ ]:
err_pct = pred.sub(y, axis=0).abs().div(y, axis=0) * 100
print("스탠드별 평균 오차율 (%)")
display(err_pct.groupby(df["stand"]).mean().round(1))
print("강종별 평균 오차율 (%)")
display(err_pct.groupby(df["grade"]).mean().round(1))

- AI 모델은 모든 스탠드에서 오차율을 2.2~3.7%로 낮췄어요. 기준선이 가장 못 맞히던 F5(11.0%)도 하이브리드는 2.8%예요.
- 기준선이 약했던 Grade09·Grade10(8.7%, 9.7%)도 하이브리드는 2.6%, 2.2%예요. 하이브리드는 모든 강종에서 3% 미만이에요.

## 7. 그래프로 보기

- 왼쪽: 모델별 MAE. 회색은 물리식, 파랑은 AI 모델이에요.
- 오른쪽: 스탠드별 **예측 - 실제** 분포를 기준선(회색)과 가장 좋은 AI 모델(파랑)로 비교해요. 0선에 붙고 상자가 좁을수록 좋아요.

In [ ]:
best = result.drop(index=["M3 (기준선)", "M3 강종별"])["MAE (MN)"].idxmin()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.4]})
order = result["MAE (MN)"].sort_values(ascending=False)
colors = [GRAY if m.startswith("M3") else BLUE for m in order.index]
bars = ax1.barh(order.index, order.values, color=colors, height=0.6)
ax1.bar_label(bars, labels=[f"{v:.2f}" for v in order.values], padding=3, color="#52514e")
ax1.set_xlabel("MAE (MN)")
ax1.set_title("모델별 평균 오차")
ax1.yaxis.grid(False)
ax1.set_xlim(0, order.max() * 1.18)

positions = np.arange(len(STANDS))
for offset, model_name, color in [(-0.18, "M3 (기준선)", GRAY), (0.18, best, BLUE)]:
    resid = pred[model_name] - y
    bp = ax2.boxplot([resid[df["stand"] == s] for s in STANDS], positions=positions + offset, widths=0.3,
                     whis=(5, 95), showfliers=False, patch_artist=True,
                     boxprops=dict(facecolor=color, edgecolor=color), medianprops=dict(color="#0b0b0b"),
                     whiskerprops=dict(color=color), capprops=dict(color=color))
    bp["boxes"][0].set_label(model_name)
ax2.axhline(0, color="#52514e", linewidth=1)
ax2.set_xticks(positions, STANDS)
ax2.xaxis.grid(False)
ax2.set_ylabel("예측 - 실제 (MN)")
ax2.set_title("스탠드별 오차 (수염 5%~95%)")
ax2.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()
print("가장 좋은 AI 모델:", best)

- 기준선(회색)은 F1을 낮게(중앙값 약 −0.6 MN), F2를 높게 예측하는 버릇이 있었는데, 하이브리드(파랑)는 모든 스탠드에서 0선에 붙어 있어요.
- 오차 폭도 크게 줄었어요. F1의 5%~95% 구간이 약 −3.4~+3.7 MN에서 약 −1.5~+1.4 MN으로 좁아졌어요.

## 8. 최적화에 쓰기 전 점검: 더 누르면 하중이 커지나?

출측 두께를 압하량의 5%만큼 더 얇게 했을 때(조금 더 누름) 예측 하중이 **줄어든** 행의 비율이에요. 물리적으로는 0%여야 해요.

In [ ]:
change = pred_more - pred
check = pd.DataFrame({
    "하중이 줄어든 행 (%)": (change < -1e-9).mean() * 100,
    "변화 없음 (%)": (change.abs() <= 1e-9).mean() * 100,
    "평균 증가량 (MN)": change.mean(),
})
display(check.style.format({"하중이 줄어든 행 (%)": "{:.1f}", "변화 없음 (%)": "{:.1f}", "평균 증가량 (MN)": "{:+.3f}"}))

- 물리식(M3)은 식 구조상 0%예요. 더 누르면 하중이 항상 커져요.
- **AI 모델은 3~5%의 행에서 더 눌렀는데 하중이 줄어든다고 예측해요**(GBDT 5.5%, 하이브리드 4.2%, MLP 3.1%).
- 예측 정확도에는 작은 문제지만 **최적화에는 큰 문제**예요. 최적화 알고리즘은 하중이 가장 작아지는 배분을 찾기 때문에, 이런 "더 누르면 오히려 가벼워지는" 잘못된 구간을 일부러 찾아가서 엉뚱한 배분을 추천할 수 있어요.
- → 다음 단계(기획서 4단계)에서 **"더 누르면 하중이 줄지 않는다"는 단조 제약**을 넣어 다시 학습해요.
- 참고: 압하량을 5% 더 늘렸을 때 하중 증가량이 물리식은 평균 0.87 MN, AI는 0.53~0.64 MN이에요. AI는 하중이 압하에 덜 민감하다고 봐요. 최적화 결과가 이 민감도에 따라 달라지므로, 최적화 단계에서 물리식과 AI 결과를 함께 비교할게요.

## 정리

| 모델 | R² | MAE (MN) | 평균 오차율 | 기준선 대비 MAE |
|---|---|---|---|---|
| M3 (기준선, 논문 모델) | 0.944 | 1.27 | 7.4% | - |
| M3 강종별 (참고) | 0.954 | 1.05 | 5.8% | −17% |
| GBDT | 0.991 | 0.46 | 2.6% | −63% |
| MLP | 0.988 | 0.55 | 3.1% | −57% |
| **하이브리드 (M3 × GBDT 보정)** | **0.992** | **0.45** | **2.5%** | **−64%** |

- **발표 문장 예시:** "논문의 물리식(R² 0.944) 대비, 물리식에 AI 보정을 더한 하이브리드 모델로 처음 보는 코일의 평균 오차를 1.27 MN에서 0.45 MN으로 64% 줄였습니다."
- **다음 단계:** 최적화에 쓰기 전에 단조 제약(더 누르면 하중이 줄지 않음)을 넣어 다시 학습하고, 정확도가 얼마나 유지되는지 확인해요.